# MILK10k — Step 9: Strong, tested explainability (v2 models, seed 42)
Four complementary levels, **each with a quantitative test**, then per-lesion evidence cards.

| Level | Question | Method | Test |
|---|---|---|---|
| A. Input | Which input (clinical / dermoscopy / metadata) drove this prediction? | **Exact Shapley values** over the 3 inputs (the model was trained with inputs missing, so every subset is in-distribution) | Efficiency check (contributions sum exactly to the prediction change) |
| B. Pixel | Where in the image? | Grad-CAM, HiResCAM, Integrated Gradients, SmoothGrad | **Deletion + insertion** vs random maps; **model-randomisation sanity check** |
| C. Concept | Which clinical concepts drive each diagnosis inside the network? | **TCAV** with MONET-defined concepts (pigmented, vessels, erythema, ulceration; artefacts as controls) | **t-test against random-concept CAVs**, Bonferroni-corrected |
| D. Case | Which confirmed cases does this resemble? | **Nearest-neighbour retrieval** of histopathology-confirmed training lesions | Neighbour-label agreement vs the model |

**Inputs:** attach the **v2 seed-42 notebook** (networks, embeddings, folds), the **notebook08 output** (v2 fusion probabilities) and **MILK10_DATASET**. Settings: **GPU T4**, **Internet On**. Runtime ≈ 40–60 min.

In [ ]:
import os, copy, time, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from itertools import combinations
from math import factorial
from PIL import Image
from scipy.stats import wilcoxon, spearmanr, ttest_ind
from sklearn.metrics import f1_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms as T
from transformers import AutoModel

def find_dir(basename, must_have):
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if os.path.basename(root) == basename and must_have in files: return root
    return None
def find_file(name):
    for root, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if name in files: return os.path.join(root, name)
    return None
RUN = find_dir("stage2v2_s42", "net_D_fold0.pt"); assert RUN, "attach the v2 seed-42 notebook (restart session if its folder is empty)"
NPZ_DIR = find_dir("v2_s42", "oof_probs_fusion.npz"); assert NPZ_DIR, "attach the committed notebook08 output"
les = pd.read_csv(f"{RUN}/milk10k_lesions_folds.csv")
IMG_DIR = os.path.dirname(find_file(les["isic_id_D"].iloc[0] + ".jpg")); assert IMG_DIR, "attach MILK10_DATASET"
OUT = "/kaggle/working/stage9"; os.makedirs(OUT, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, "\nrun:", RUN, "\nfusion probs:", NPZ_DIR, "\nimages:", IMG_DIR)

CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
ACTIONABLE = ["AKIEC", "BCC", "MAL_OTH", "MEL", "SCCKA"]; is_act_cls = np.isin(CLASSES, ACTIONABLE)
y = les["y"].values.astype(int); fold = les["fold"].values; n = len(y)
MODS = ["C", "D", "M"]; PATTERNS = ["CDM", "CD", "CM", "DM", "C", "D", "M"]
CFG = {"D": dict(size=384, backbone="facebook/convnextv2-tiny-22k-384"),
       "C": dict(size=224, backbone="facebook/convnextv2-tiny-22k-224")}
RNG = np.random.RandomState(0)

## A. Exact Shapley values over the three inputs

In [ ]:
npz = np.load(f"{NPZ_DIR}/oof_probs_fusion.npz")
P = {p: npz[f"late_drop|{p}"].astype(float) for p in PATTERNS}
prior = np.zeros((n, K))                                  # value of the empty set = class prior of the training folds
for f in range(5): prior[fold == f] = np.bincount(y[fold != f], minlength=K) / (fold != f).sum()
def value(S, cls):
    name = "".join(m for m in "CDM" if m in S)
    return (P[name] if name else prior)[np.arange(n), cls]

def shapley(cls):
    phi = {m: np.zeros(n) for m in MODS}
    for m in MODS:
        others = [o for o in MODS if o != m]
        for r in range(3):
            for S in combinations(others, r):
                w = factorial(r) * factorial(2 - r) / factorial(3)
                phi[m] += w * (value(set(S) | {m}, cls) - value(set(S), cls))
    return phi
pred = P["CDM"].argmax(1)
PHI = shapley(pred)                                       # contributions to the probability of the predicted class
assert np.allclose(sum(PHI.values()), value(set(MODS), pred) - value(set(), pred)), "efficiency check failed"
print("Efficiency check passed: contributions sum exactly to p(full) - p(prior) for every lesion.")

ok = pred == y
sh = pd.DataFrame({f"phi_{m}": pd.Series(PHI[m][ok]).groupby(y[ok]).mean() for m in MODS})
dom = np.argmax(np.column_stack([PHI[m] for m in MODS]), 1)
sh["dermoscopy_dominant"] = pd.Series((dom == 1)[ok]).groupby(y[ok]).mean()
sh["clinical_dominant"] = pd.Series((dom == 0)[ok]).groupby(y[ok]).mean()
sh["n_correct"] = pd.Series(y[ok]).value_counts().reindex(sh.index).values
sh.index = [CLASSES[k] for k in sh.index]; sh.to_csv(f"{OUT}/A_shapley_by_class.csv")
sh.round(3)

In [ ]:
# Does the dermoscopy contribution to the TRUE class match the actual benefit of adding dermoscopy?
PHI_true = shapley(y); eps = 1e-9
gain = np.log(P["CDM"][np.arange(n), y] + eps) - np.log(P["CM"][np.arange(n), y] + eps)
print("Spearman(phi_D for true class, log-gain CM->CDM):", round(spearmanr(PHI_true["D"], gain)[0], 3))
fig, ax = plt.subplots(figsize=(8, 4))
sh[["phi_C", "phi_D", "phi_M"]].plot.bar(ax=ax, color=["#d08c60", "#5b7db1", "#8aa66a"])
ax.set_ylabel("mean Shapley contribution to p(predicted class)"); ax.set_title("Which input drives each correct diagnosis")
ax.legend(["clinical photo", "dermoscopy", "metadata"]); plt.tight_layout(); plt.savefig(f"{OUT}/A_fig_shapley.png", dpi=200); plt.show()

## D. Case-based retrieval (CPU): nearest histopathology-confirmed training lesions

In [ ]:
def l2n(a): return a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
EMB = {(m, f): np.load(f"{RUN}/emb_{m}_fold{f}.npy").astype(np.float32) for m in "CD" for f in range(5)}
def retrieve(space, k=5):
    nn_idx, nn_sim = np.zeros((n, k), int), np.zeros((n, k))
    for f in range(5):
        E = l2n(np.concatenate([l2n(EMB[(m, f)]) for m in space], 1))
        q, g = np.where(fold == f)[0], np.where(fold != f)[0]
        S = E[q] @ E[g].T; top = np.argsort(-S, 1)[:, :k]
        nn_idx[q], nn_sim[q] = g[top], np.take_along_axis(S, top, 1)
    return nn_idx, nn_sim
rows = []
for space, name in [(["C"], "clinical only"), (["D"], "dermoscopy only"), (["C", "D"], "clinical + dermoscopy")]:
    idx, sim = retrieve(space)
    nb_lab = y[idx]; vote = np.array([np.bincount(r, minlength=K).argmax() for r in nb_lab])
    rows.append(dict(space=name, precision_at_5=(nb_lab == y[:, None]).mean(),
                     kNN_macro_f1=f1_score(y, vote, average="macro", labels=range(K), zero_division=0),
                     kNN_bal_acc=balanced_accuracy_score(y, vote),
                     actionable_agreement=(is_act_cls[nb_lab] == is_act_cls[y][:, None]).mean()))
    if space == ["C", "D"]: NN_IDX, NN_SIM = idx, sim
ret = pd.DataFrame(rows).set_index("space")
ret.loc["(model: fusion, all inputs)"] = [np.nan, f1_score(y, pred, average="macro", labels=range(K), zero_division=0),
                                          balanced_accuracy_score(y, pred), np.nan]
ret.to_csv(f"{OUT}/D_retrieval.csv"); ret.round(3)

## Networks, preprocessing (identical to training v2: resize + Shades-of-Gray) and four attribution methods

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
def shades_of_gray(im, p=6):
    a = np.asarray(im).astype(np.float32); il = np.power(np.mean(np.power(a, p), axis=(0, 1)), 1 / p)
    il = il / (np.sqrt((il ** 2).sum()) + 1e-6); return Image.fromarray(np.clip(a / (il * np.sqrt(3) + 1e-6), 0, 255).astype(np.uint8))
def prep_pil(i, mod):
    s = CFG[mod]["size"]; short = int(s * 1.15)
    im = Image.open(f"{IMG_DIR}/{les[f'isic_id_{mod}'].iloc[i]}.jpg").convert("RGB"); w_, h_ = im.size; r = short / min(w_, h_)
    return shades_of_gray(im.resize((max(short, round(w_ * r)), max(short, round(h_ * r))), Image.BICUBIC))
TF = {m: T.Compose([T.Resize((CFG[m]["size"],) * 2), T.ToTensor(), T.Normalize(MEAN, STD)]) for m in "CD"}
def load_x(i, mod): return TF[mod](prep_pil(i, mod)).unsqueeze(0).to(device)
def load_rgb(i, mod): s = CFG[mod]["size"]; return np.asarray(prep_pil(i, mod).resize((s, s)))

class Net(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(backbone)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.backbone.config.hidden_sizes[-1], K))
    def embed(self, x): return self.backbone(pixel_values=x).pooler_output
    def forward(self, x): return self.head(self.embed(x))
def load_net(mod, f):
    net = Net(CFG[mod]["backbone"]).to(device)
    net.load_state_dict(torch.load(f"{RUN}/net_{mod}_fold{f}.pt", map_location=device)); return net.eval()

BLUR = T.GaussianBlur(51, sigma=20)
def _norm(m): m = m - m.min(); return m / (m.max() + 1e-8)
def _smooth(m): return F.avg_pool2d(m, 11, stride=1, padding=5)

class Explainer:
    """Grad-CAM / HiResCAM on the last ConvNeXt stage; Integrated Gradients and SmoothGrad on the input."""
    def __init__(self, net):
        for mod_ in net.modules(): mod_._forward_hooks.clear()
        self.net = net; self.act = self.grad = None
        net.backbone.encoder.stages[-1].register_forward_hook(self._fwd)
    def _fwd(self, module, inp, out):
        o = out[0] if isinstance(out, (tuple, list)) else out; self.act = o
        if o.requires_grad: o.register_hook(lambda g: setattr(self, "grad", g))
    def cams(self, x, target):
        self.net.zero_grad(set_to_none=True)
        with torch.enable_grad(): self.net(x)[0, target].backward()
        A, G = self.act.detach(), self.grad.detach(); H, W = x.shape[-2:]
        up = lambda m: _norm(F.interpolate(m, size=(H, W), mode="bilinear", align_corners=False)[0, 0])
        return {"Grad-CAM": up(F.relu((G.mean((2, 3), keepdim=True) * A).sum(1, keepdim=True))),
                "HiResCAM": up(F.relu((G * A).sum(1, keepdim=True)))}
    def _input_grad(self, xb, target):
        xb = xb.detach().clone().requires_grad_(True)
        with torch.enable_grad(): g, = torch.autograd.grad(self.net(xb)[:, target].sum(), xb)
        return g
    def integrated_gradients(self, x, target, steps=32):
        x0 = BLUR(x); al = torch.linspace(1 / steps, 1, steps, device=x.device)
        g = torch.cat([self._input_grad(x0 + a[:, None, None, None] * (x - x0), target) for a in al.split(8)]).mean(0, keepdim=True)
        return _norm(_smooth(((x - x0) * g).abs().sum(1, keepdim=True))[0, 0])
    def smoothgrad(self, x, target, n_samples=16, sigma=0.15):
        s = sigma * (x.max() - x.min()); xr = x.repeat(8, 1, 1, 1)
        g = torch.cat([self._input_grad(xr + torch.randn_like(xr) * s, target).abs() for _ in range(n_samples // 8)])
        return _norm(_smooth(g.mean(0, keepdim=True).sum(1, keepdim=True))[0, 0])
    def all_maps(self, x, target):
        m = self.cams(x, target)
        m["Integrated Gradients"] = self.integrated_gradients(x, target); m["SmoothGrad"] = self.smoothgrad(x, target)
        return m
METHODS = ["Grad-CAM", "HiResCAM", "Integrated Gradients", "SmoothGrad"]

## B1. Faithfulness benchmark: deletion (lower = better) and insertion (higher = better) vs a random smooth map

In [ ]:
STEPS = np.linspace(0, 1, 11); TRAPZ = getattr(np, "trapezoid", None) or np.trapz
@torch.no_grad()
def del_ins(net, x, rank_map, target):
    H, W = rank_map.shape; N = H * W; xb = x[0]; bl = BLUR(x)[0]; order = torch.argsort(rank_map.flatten(), descending=True)
    masks = []
    for p in STEPS:
        m = torch.zeros(N, device=x.device); m[order[:int(round(p * N))]] = 1; masks.append(m.view(1, H, W))
    M = torch.stack(masks)
    pd_ = net(xb * (1 - M) + bl * M).softmax(1)[:, target].cpu().numpy()          # deletion: blur most important first
    pi_ = net(bl * (1 - M) + xb * M).softmax(1)[:, target].cpu().numpy()          # insertion: reveal most important first
    return TRAPZ(pd_, STEPS), TRAPZ(pi_, STEPS)
def random_map(seed, H):
    g = torch.Generator().manual_seed(int(seed)); r = torch.rand(1, 1, H // 32, H // 32, generator=g)
    return F.interpolate(r, size=(H, H), mode="bilinear", align_corners=False)[0, 0].to(device)

N_PER_FOLD = 12; rows = []; t0 = time.time()
for mod in ["D", "C"]:
    for f in range(5):
        net = load_net(mod, f); ex = Explainer(net)
        for i in RNG.choice(np.where(fold == f)[0], N_PER_FOLD, replace=False):
            x = load_x(i, mod)
            with torch.no_grad(): tgt = int(net(x).argmax(1))
            maps = ex.all_maps(x, tgt); maps["random"] = random_map(i, x.shape[-1])
            for meth, m in maps.items():
                d_, i_ = del_ins(net, x, m, tgt)
                rows.append(dict(modality=mod, fold=f, lesion=i, method=meth, deletion=d_, insertion=i_))
        del net, ex; torch.cuda.empty_cache(); print(f"{mod} fold {f} done ({time.time() - t0:.0f}s)")
bench = pd.DataFrame(rows); bench.to_csv(f"{OUT}/B1_faithfulness_long.csv", index=False)

out = []
for (mod, meth), d in bench[bench.method != "random"].groupby(["modality", "method"]):
    r = bench[(bench.modality == mod) & (bench.method == "random")].set_index("lesion").loc[d["lesion"]]
    out.append(dict(modality={"C": "clinical", "D": "dermoscopy"}[mod], method=meth,
                    deletion=d["deletion"].mean(), insertion=d["insertion"].mean(),
                    ins_minus_del=(d["insertion"] - d["deletion"]).mean(),
                    beats_random_deletion=(d["deletion"].values < r["deletion"].values).mean(),
                    p_vs_random=wilcoxon(d["deletion"].values, r["deletion"].values).pvalue))
rnd = bench[bench.method == "random"].groupby("modality")[["deletion", "insertion"]].mean()
faith = pd.DataFrame(out).set_index(["modality", "method"]); faith.to_csv(f"{OUT}/B1_faithfulness_summary.csv")
print("Random-map reference (deletion / insertion):", rnd.round(3).to_dict("index"))
faith.round(3)

## B2. Model-randomisation sanity check (Adebayo et al., 2018)
Top layers are re-initialised step by step; a map that **barely changes** (high rank correlation with the original) is not explaining the trained model. Fold-0 networks, 20 lesions per image type.

In [ ]:
def randomized_copy(net, level):
    r = copy.deepcopy(net); mods = [r.head]
    if level >= 2: mods += [r.backbone.encoder.stages[-1], r.backbone.layernorm]
    if level >= 3: mods += [r.backbone.encoder.stages[-2]]
    g = torch.Generator().manual_seed(level)
    with torch.no_grad():
        for m in mods:
            for p in m.parameters(): p.copy_((torch.randn(p.shape, generator=g) * 0.02).to(p.device))
    return r.eval()
def small(m): return F.adaptive_avg_pool2d(m[None, None], 56)[0, 0].flatten().cpu().numpy()
LEVELS = {1: "head", 2: "head + last stage", 3: "head + last 2 stages"}
rows = []
for mod in ["D", "C"]:
    net = load_net(mod, 0); ex = Explainer(net)
    rand = {lv: Explainer(randomized_copy(net, lv)) for lv in LEVELS}
    for i in RNG.choice(np.where(fold == 0)[0], 20, replace=False):
        x = load_x(i, mod)
        with torch.no_grad(): tgt = int(net(x).argmax(1))
        orig = ex.all_maps(x, tgt)
        for lv, rex in rand.items():
            rm = rex.all_maps(x, tgt)
            for meth in METHODS:
                rows.append(dict(modality=mod, level=LEVELS[lv], method=meth, spearman=spearmanr(small(orig[meth]), small(rm[meth]))[0]))
    del net, ex, rand; torch.cuda.empty_cache()
san = pd.DataFrame(rows).groupby(["modality", "method", "level"])["spearman"].mean().unstack("level")[list(LEVELS.values())]
san.to_csv(f"{OUT}/B2_sanity_check.csv")
print("Rank correlation between original and randomised-model maps (lower = passes the sanity check):"); san.round(3)

## C. TCAV with MONET concepts (dermoscopy network, fold 0)
Concept examples = training lesions in the top 15% of each **dermoscopic** MONET score; CAVs are learned on stage-3 activations against 20 random sets, and compared with 20 random-vs-random CAVs (two-sided t-test, Bonferroni).

In [ ]:
CONCEPTS = {"pigmented": "MONET_pigmented_D", "vessels": "MONET_vasculature_vessels_D", "erythema": "MONET_erythema_D",
            "ulceration/crust": "MONET_ulceration_crust_D", "hair (control)": "MONET_hair_D",
            "gel/fluid (control)": "MONET_gel_water_drop_fluid_dermoscopy_liquid_D",
            "pen marking (control)": "MONET_skin_markings_pen_ink_purple_pen_D"}
N_CONCEPT, N_RANDOM_SETS, N_POOL = 150, 20, 1000
net = load_net("D", 0); act_store = {}
for mod_ in net.modules(): mod_._forward_hooks.clear()
net.backbone.encoder.stages[-2].register_forward_hook(lambda m, i, o: act_store.__setitem__("a", o[0] if isinstance(o, (tuple, list)) else o))

train_idx = np.where(fold != 0)[0]
pos = {c: train_idx[np.argsort(-les[col].values[train_idx])[:int(0.15 * len(train_idx))]] for c, col in CONCEPTS.items()}
pos = {c: RNG.choice(v, N_CONCEPT, replace=False) for c, v in pos.items()}
pool = RNG.choice(train_idx, N_POOL, replace=False)
need = np.unique(np.concatenate(list(pos.values()) + [pool]))
ACT = {}
with torch.no_grad():
    for k_, i in enumerate(need):
        net(load_x(i, "D")); ACT[i] = act_store["a"].mean((2, 3))[0].float().cpu().numpy()
print("activations for", len(ACT), "lesions")

def cav(pos_ids, neg_ids):
    X = np.stack([ACT[i] for i in np.concatenate([pos_ids, neg_ids])]); lab = np.r_[np.ones(len(pos_ids)), np.zeros(len(neg_ids))]
    sc = StandardScaler().fit(X); lr = LogisticRegression(C=0.1, max_iter=2000).fit(sc.transform(X), lab)
    v = lr.coef_[0] / sc.scale_; return v / np.linalg.norm(v), lr.score(sc.transform(X), lab)
CAVS, CAV_ACC = {}, {}
for c in CONCEPTS:
    negpool = np.setdiff1d(pool, pos[c]); res_ = [cav(pos[c], RNG.choice(negpool, N_CONCEPT, replace=False)) for _ in range(N_RANDOM_SETS)]
    CAVS[c] = np.stack([r[0] for r in res_]); CAV_ACC[c] = np.mean([r[1] for r in res_])
NULL = np.stack([cav(*np.split(RNG.choice(pool, 2 * N_CONCEPT, replace=False), 2))[0] for _ in range(N_RANDOM_SETS)])
print("CAV separability (train accuracy):", {c: round(a, 3) for c, a in CAV_ACC.items()})

In [ ]:
# gradients of every class logit w.r.t. stage-3 activations, summed over positions (directional derivative along a CAV)
test_idx = np.where(fold == 0)[0]
sel = np.concatenate([RNG.choice(test_idx[y[test_idx] == k], min(40, (y[test_idx] == k).sum()), replace=False) for k in range(K)])
G = np.zeros((len(sel), K, CAVS["pigmented"].shape[1])); t0 = time.time()
for j, i in enumerate(sel):
    with torch.enable_grad():
        logits = net(load_x(i, "D")); a = act_store["a"]
        for k in range(K):
            g, = torch.autograd.grad(logits[0, k], a, retain_graph=k < K - 1); G[j, k] = g[0].sum((1, 2)).float().cpu().numpy()
print(f"gradients for {len(sel)} lesions in {time.time() - t0:.0f}s")
ysel = y[sel]
def tcav_scores(V):            # V: (R, C) -> (R, K) fraction of class-k lesions with positive directional derivative
    return np.stack([[(G[ysel == k, k] @ v > 0).mean() if (ysel == k).sum() else np.nan for k in range(K)] for v in V])
null_s = tcav_scores(NULL); rows = []
valid_k = [k for k in range(K) if (ysel == k).sum() >= 10]
for c in CONCEPTS:
    s = tcav_scores(CAVS[c])
    for k in valid_k:
        p = ttest_ind(s[:, k], null_s[:, k]).pvalue
        rows.append(dict(concept=c, cls=CLASSES[k], tcav=s[:, k].mean(), null=null_s[:, k].mean(), p=p))
tc = pd.DataFrame(rows); tc["significant"] = tc["p"] < 0.05 / len(tc); tc.to_csv(f"{OUT}/C_tcav.csv", index=False)
heat = tc.pivot(index="concept", columns="cls", values="tcav").reindex(list(CONCEPTS))
sig = tc.pivot(index="concept", columns="cls", values="significant").reindex(list(CONCEPTS)).fillna(False).astype(bool)
fig, ax = plt.subplots(figsize=(10, 4.5)); im = ax.imshow(np.where(sig, heat, np.nan), cmap="RdBu_r", vmin=0, vmax=1, aspect="auto")
for (r_, c_), v in np.ndenumerate(heat.values):
    ax.text(c_, r_, f"{v:.2f}" + ("*" if sig.values[r_, c_] else ""), ha="center", va="center", fontsize=7, color="k" if sig.values[r_, c_] else "gray")
ax.set_xticks(range(heat.shape[1])); ax.set_xticklabels(heat.columns); ax.set_yticks(range(len(heat))); ax.set_yticklabels(heat.index)
plt.colorbar(im, label="TCAV score (coloured only if significant)")
ax.set_title(f"TCAV, dermoscopy network: * = significant vs random CAVs (Bonferroni, {len(tc)} tests); classes with ≥10 test lesions")
plt.tight_layout(); plt.savefig(f"{OUT}/C_fig_tcav.png", dpi=200); plt.show()
print("Classes with fewer than 10 test lesions in fold 0 are omitted:", [CLASSES[k] for k in range(K) if k not in valid_k])

## E. Evidence cards (randomly chosen, fold 0): all four levels for one lesion

In [ ]:
best = faith.xs("dermoscopy", level="modality")["ins_minus_del"].idxmax(); print("map method used on cards (best on dermoscopy):", best)
for mod_ in net.modules(): mod_._forward_hooks.clear()
exD = Explainer(net)
def pick(cond): c_ = np.where(cond & (fold == 0))[0]; return int(RNG.choice(c_)) if len(c_) else None
mel = CLASSES.index("MEL")
cases = {"melanoma, correct": pick((y == mel) & (pred == mel)), "melanoma, missed": pick((y == mel) & ~is_act_cls[pred]),
         "BCC, correct": pick((y == 1) & (pred == 1)), "vascular, correct": pick((y == 10) & (pred == 10))}
for title, i in cases.items():
    if i is None: continue
    x = load_x(i, "D"); maps = exD.all_maps(x, int(pred[i])) if best in ("Integrated Gradients", "SmoothGrad") else exD.cams(x, int(pred[i]))
    fig = plt.figure(figsize=(18, 3.6)); gs = fig.add_gridspec(1, 8, width_ratios=[1, 1, 1, 1.2, 1.3, .8, .8, .8])
    ax = [fig.add_subplot(gs[0, j]) for j in range(8)]
    ax[0].imshow(load_rgb(i, "C")); ax[0].set_title("clinical", fontsize=8)
    ax[1].imshow(load_rgb(i, "D")); ax[1].set_title("dermoscopy", fontsize=8)
    ax[2].imshow(load_rgb(i, "D")); ax[2].imshow(maps[best].cpu().numpy(), cmap="jet", alpha=0.45); ax[2].set_title(best, fontsize=8)
    ax[3].bar(["clinical", "dermo.", "meta"], [PHI[m][i] for m in MODS], color=["#d08c60", "#5b7db1", "#8aa66a"])
    ax[3].axhline(0, color="k", lw=0.5); ax[3].set_title(f"Shapley → p({CLASSES[pred[i]]})", fontsize=8); ax[3].tick_params(labelsize=7)
    top_cm = np.argsort(-P["CM"][i])[:3]; top_all = np.argsort(-P["CDM"][i])[:3]
    txt = ("Clinical + metadata:\n" + "\n".join(f"  {CLASSES[k]} {P['CM'][i, k]:.2f}" for k in top_cm)
           + "\nAll inputs:\n" + "\n".join(f"  {CLASSES[k]} {P['CDM'][i, k]:.2f}" for k in top_all)
           + "\nDermoscopic concepts:\n" + "\n".join(f"  {c} {les[col].iloc[i]:.2f}" for c, col in list(CONCEPTS.items())[:4]))
    ax[4].text(0, 1, txt, va="top", fontsize=7.5, family="monospace"); ax[4].set_title("probabilities & concepts", fontsize=8)
    for j in range(3):
        nb_i = NN_IDX[i, j]; ax[5 + j].imshow(load_rgb(nb_i, "D")); ax[5 + j].set_title(f"neighbour {j+1}: {CLASSES[y[nb_i]]}\nsim {NN_SIM[i, j]:.2f}", fontsize=7)
    for a_ in ax: a_.set_xticks([]) if a_ is not ax[3] else None
    for a_ in [ax[0], ax[1], ax[2], ax[4], ax[5], ax[6], ax[7]]: a_.axis("off")
    fig.suptitle(f"{title} | true {CLASSES[y[i]]} | predicted {CLASSES[pred[i]]} (lesion {les['lesion_id'].iloc[i]})", fontsize=10)
    plt.tight_layout(); fname = title.replace(", ", "_").replace(" ", "_"); plt.savefig(f"{OUT}/E_card_{fname}.png", dpi=170); plt.show()